# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [ ]:
!pip install -q torch bitsandbytes datasets accelerate loralib


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import torch
import torch.nn as nn

In [ ]:
print(torch.__version__)
print(torch.cuda.is_available())

2.8.0+cu129
True


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig
import os, bitsandbytes as bnb

In [ ]:
model_name = "google/gemma-2-9b-it"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [ ]:
i = 5
for p in model.named_parameters():
  print(p)
  i -= 1
  if i < 0:
    break

('model.embed_tokens.weight', Parameter containing:
tensor([[-0.0510,  0.0106, -0.0422,  ..., -0.0294, -0.0352,  0.0127],
        [-0.0007,  0.0127,  0.0083,  ...,  0.0049, -0.0027, -0.0275],
        [-0.0112,  0.0026,  0.0081,  ..., -0.0006,  0.0043,  0.0033],
        ...,
        [-0.0264,  0.0157,  0.0041,  ..., -0.0588, -0.0361, -0.0080],
        [-0.0522,  0.0139, -0.0004,  ..., -0.0061, -0.0491, -0.0109],
        [-0.0413,  0.0264, -0.0249,  ...,  0.0015, -0.0315, -0.0033]],
       device='cuda:0', dtype=torch.float16, requires_grad=True))
('model.layers.0.self_attn.q_proj.weight', Parameter containing:
Parameter(Int8Params([[-15, -27,  48,  ...,   6,  15,   8],
            [-42, -30, -28,  ...,   6, -16, -49],
            [-68, -30, -52,  ...,   8,  13, -36],
            ...,
            [ 15,  44,  21,  ..., -14,  18, -16],
            [-52,   2, -20,  ...,  51, -16,  -4],
            [ -4, -20,   0,  ...,  10,   2, -21]], device='cuda:0',
           dtype=torch.int8)))
('model

In [ ]:
for param in model.parameters():
  param.requires_grad = False
  if param.ndim == 1:
    param.data = param.data.to(torch.float32)

In [ ]:
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

In [ ]:
class CastOutputToFloat(nn.Sequential):
  def forward(self, x):
    return super().forward(x).to(torch.float32)

model.lm_head = CastOutputToFloat(model.lm_head)

In [ ]:
def print_trainable_parameters(model):
    """
  printing the number of trainable paramters in the model
  """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}")

In [ ]:
message = "Be yourself; everyone else is already taken."
text_ids = tokenizer(message, return_tensors="pt").to(model.device)

In [ ]:
with torch.no_grad():
    res = model.generate(
        **text_ids,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
    )

In [ ]:
print(tokenizer.decode(res[0], skip_special_tokens=True))

Be yourself; everyone else is already taken.

-Oscar Wilde-

This quote, like many of Wilde's work, is both witty and profound. It encourages us to embrace our individuality, to celebrate what makes us unique, rather than trying to be someone we're not. 


## Fine tuning

In [ ]:
pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 127.9 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["k_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
print_trainable_parameters(model)

trainable params: 0 || all params: 9241705984 || trainable%: 0.0


In [ ]:
lora_model = get_peft_model(model, config)
print_trainable_parameters(lora_model)

/usr/local/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 7569408 || all params: 9249275392 || trainable%: 0.08183784868755263


In [ ]:
import transformers
from datasets import load_dataset

In [ ]:
data = load_dataset("Abirate/english_quotes")

README.md: 0.00B [00:00, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [ ]:
data

DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags'],
        num_rows: 2508
    })
})

In [ ]:
data['train'][0]

{'quote': '“Be yourself; everyone else is already taken.”',
 'author': 'Oscar Wilde',
 'tags': ['be-yourself',
  'gilbert-perreira',
  'honesty',
  'inspirational',
  'misattributed-oscar-wilde',
  'quote-investigator']}

In [ ]:
def merge_columns(example):
  example['prediction'] = example['quote'] + " ->: " + str(example['tags'])
  return example

In [ ]:
data['train'] = data['train'].map(merge_columns)
data['train']['prediction'][0]

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

"“Be yourself; everyone else is already taken.” ->: ['be-yourself', 'gilbert-perreira', 'honesty', 'inspirational', 'misattributed-oscar-wilde', 'quote-investigator']"

In [ ]:
data = data.map(lambda samples: tokenizer(samples['prediction']), batched=True)

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [ ]:
data

DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags', 'prediction', 'input_ids', 'attention_mask'],
        num_rows: 2508
    })
})

In [ ]:
lora_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 3584, padding_idx=0)
        (layers): ModuleList(
          (0-41): 42 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): Linear8bitLt(in_features=3584, out_features=4096, bias=False)
              (k_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=3584, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): Para

In [ ]:
from peft import PeftModel

# Check 1
print("Is PeftModel:", isinstance(lora_model, PeftModel))

# Check 2 - correct import path for newer transformers
from transformers.utils import is_peft_available
print("PEFT available:", is_peft_available())

# Check 3 - replicate Trainer's exact check
from peft import PeftMixedModel
def _is_peft_model(model):
    if is_peft_available():
        classes_to_check = (PeftModel, PeftMixedModel)
        return isinstance(model, classes_to_check)
    return False

print("Trainer _is_peft_model:", _is_peft_model(lora_model))

# Check 4 - what quantization attributes exist
base = lora_model.base_model.model
print("base type:", type(base))
print("has hf_quantizer:", hasattr(base, 'hf_quantizer'))
print("quantization_config:", getattr(base.config, 'quantization_config', 'NOT SET'))

# Check 5 - type chain
print("type(lora_model):", type(lora_model))

Is PeftModel: True
PEFT available: False
Trainer _is_peft_model: False
base type: <class 'transformers.models.gemma2.modeling_gemma2.Gemma2ForCausalLM'>
has hf_quantizer: True
quantization_config: BitsAndBytesConfig {
  "_load_in_4bit": false,
  "_load_in_8bit": true,
  "bnb_4bit_compute_dtype": "float32",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "fp4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": false,
  "load_in_8bit": true,
  "quant_method": "bitsandbytes"
}

type(lora_model): <class 'peft.peft_model.PeftModelForCausalLM'>


In [ ]:
import importlib
import transformers.utils
import transformers.utils.import_utils as iu

# Force re-check of peft availability
iu._peft_available = None  # clear the cached result

# Monkey-patch is_peft_available to always return True
# since we've confirmed peft IS installed
import peft  # confirm it imports fine
transformers.utils.is_peft_available = lambda: True

# Also patch it in trainer's local scope
import transformers.trainer as trainer_module
trainer_module.is_peft_available = lambda: True

# Verify
from transformers.utils import is_peft_available
print("PEFT available now:", is_peft_available())

PEFT available now: True


In [ ]:
import transformers.trainer as trainer_module
from peft import PeftModel, PeftMixedModel

# Inject the missing names into trainer's module scope
trainer_module.PeftModel = PeftModel
trainer_module.PeftMixedModel = PeftMixedModel

print("Patched successfully")

Patched successfully


In [ ]:
model.config.use_cache = False

trainer = transformers.Trainer(
    model=lora_model,
    train_dataset=data['train'],
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,   # effective batch = 4
        warmup_steps=10,
        max_steps=50,
        learning_rate=2e-4,
        fp16=False,
        bf16=True,                       # Gemma2 prefers bf16 over fp16
        logging_steps=1,
        output_dir='outputs',
        optim="paged_adamw_8bit",        # needed for 8-bit quantized base
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
It is strongly recommended to train Gemma2 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.
/usr/local/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
1,3.651800
2,2.713200
3,3.512900
4,2.904500
5,3.018700
6,1.919200
7,2.873100
8,2.035500
9,2.822100
10,2.776700


TrainOutput(global_step=50, training_loss=1.787513986825943, metrics={'train_runtime': 218.6666, 'train_samples_per_second': 0.915, 'train_steps_per_second': 0.229, 'total_flos': 565294025404416.0, 'train_loss': 1.787513986825943, 'epoch': 0.07974481658692185})

In [ ]:
lora_model.push_to_hub("afreediz/gemma-2-9b-it-tagger-test-adapter",
                      commit_message = "Testing Lora Training method",
                      private=False)

## Inference

In [ ]:
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
peft_model_id = "afreediz/gemma-2-9b-it-tagger-test-adapter"
config = PeftConfig.from_pretrained(peft_model_id)
model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path,
                                            return_dict = True,
                                            load_in_8bit = True,
                                            device_map = 'auto')
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

model = PeftModel.from_pretrained(model,peft_model_id)

adapter_config.json: 0.00B [00:00, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/30.3M [00:00<?, ?B/s]

In [ ]:
batch = tokenizer("“Be yourself; everyone else is already taken.” ->: ", return_tensors='pt').to(model.device)

#with torch.cuda.amp.autocast():
#  output_tokens = model.generate(**batch, max_new_tokens=50)

# USE ABOVE COMMENTED CODE IF BELOW LINE OF CODE IS NOT WORKING AS EXPECTED
output_tokens = model.generate(**batch, max_new_tokens=50)

print('\n\n', tokenizer.decode(output_tokens[0], skip_special_tokens=True))



 “Be yourself; everyone else is already taken.” ->: 
: ['be-yourself', 'inspirational', 'inspirational-quotes', 'inspirational-quotes-about-life', 'inspirational-quotes-about-life-and-love', 'inspirational-quotes-about-life-and-living',
